In [ ]:
#Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#Load the Spotify songs dataset
#Make sure spotify_songs.csv is uploaded in the same folder/session as this notebook
df = pd.read_csv('spotify_songs.csv')

#Clean column names so they are easier to use in Pandas
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

#Remove duplicate rows if any exist
df = df.drop_duplicates()

#Create a new column that shows streams in millions for easier reading
df['streams_millions'] = df['streams'] / 1000000

#Display the first few rows of the dataset
df.head()


Problem Space

Music streaming has changed the way songs become popular and how success is measured in the music industry. Instead of only looking at album sales or radio play, Spotify-style data allows us to study streams, playlists, chart placement, and audio features such as danceability, energy, valence, acousticness, and speechiness.

My analysis seeks to determine patterns in Spotify song success by addressing the following questions:
1. Which songs have the most Spotify streams?
2. Which artists have the highest total streams?
3. Which genres have the highest average popularity?
4. How do streams change by year?
5. Is playlist placement related to streams?
6. Do audio features like danceability and energy help explain popularity?


Questions to Explore

As mentioned previously, the following questions will guide my analysis:
1. Which songs have the most Spotify streams?
2. Which artists have the highest total streams?
3. Which genres have the highest average popularity?
4. How do streams change by year?
5. Does being placed in more Spotify playlists relate to having more streams?
6. How do danceability and energy relate to popularity?


In [ ]:
#Analyzing the Top Songs by Spotify Streams
#Using Pandas to sort the dataset by streams in millions
top_songs = (
    df[['song', 'artist', 'year', 'streams_millions', 'popularity']]
    .sort_values('streams_millions', ascending=False)
    .head(10)
)

#Display the Pandas table
display(top_songs)

#Create a bar chart of the top streamed songs
plt.figure(figsize=(10,6))
plt.barh(top_songs['song'], top_songs['streams_millions'])
plt.gca().invert_yaxis()
plt.title('Top 10 Songs by Spotify Streams')
plt.xlabel('Streams in Millions')
plt.ylabel('Song')
plt.show()


Analysis of the Top Songs by Spotify Streams

The chart shows which songs had the highest number of streams in the dataset. Songs such as *All of Me*, *Closer*, and *Beautiful People* appear near the top, showing that highly streamed songs can come from different years and artists. This suggests that Spotify success is not only about being a newer song, but also about replay value, artist recognition, and long-term popularity.


In [ ]:
#Analyzing Total Streams by Artist
#Using groupby to combine all songs for each artist
top_artists = (
    df.groupby('artist', as_index=False)
    .agg(
        total_streams_millions=('streams_millions', 'sum'),
        song_count=('song', 'count'),
        avg_popularity=('popularity', 'mean')
    )
    .sort_values('total_streams_millions', ascending=False)
    .head(10)
)

#Display the Pandas table
display(top_artists)

#Create a bar chart of artists by total streams
plt.figure(figsize=(10,6))
plt.barh(top_artists['artist'], top_artists['total_streams_millions'])
plt.gca().invert_yaxis()
plt.title('Top 10 Artists by Total Streams')
plt.xlabel('Total Streams in Millions')
plt.ylabel('Artist')
plt.show()


Analysis of the Correlation Between Artist and Streams

The graph shows that artists with multiple popular songs have an advantage because their total streams add across their catalog. This makes artist presence important when studying music streaming data. The results suggest that Spotify success is partly connected to the strength of the artist's overall catalog, not just one individual song.


In [ ]:
#Analyzing Popularity by Genre
#Using Pandas groupby to calculate average popularity and average streams by genre
genre_summary = (
    df.groupby('genre')
    .agg(
        avg_popularity=('popularity', 'mean'),
        avg_streams_millions=('streams_millions', 'mean'),
        song_count=('song', 'count')
    )
    .reset_index()
    .sort_values('avg_popularity', ascending=False)
)

top_genres = genre_summary.head(10)

#Display the Pandas table
display(top_genres)

#Create a bar chart of average popularity by genre
plt.figure(figsize=(10,6))
plt.barh(top_genres['genre'], top_genres['avg_popularity'])
plt.gca().invert_yaxis()
plt.title('Average Popularity by Genre')
plt.xlabel('Average Popularity')
plt.ylabel('Genre')
plt.show()


Genre Popularity Analysis

The chart above shows that some genres have higher average popularity than others. However, the `song_count` column is important because a genre with only one or two songs can have a high average if those songs are very successful. This means genre popularity should be interpreted carefully, since small groups can be strongly affected by one major hit.


In [ ]:
#Analyzing Streams by Year
#Grouping by year to see how total streams and average popularity change over time
year_summary = (
    df.groupby('year')
    .agg(
        total_streams_millions=('streams_millions', 'sum'),
        avg_streams_millions=('streams_millions', 'mean'),
        avg_popularity=('popularity', 'mean'),
        song_count=('song', 'count')
    )
    .reset_index()
    .sort_values('year')
)

#Display the Pandas table
display(year_summary)

#Create a line chart of total streams by year
plt.figure(figsize=(10,6))
plt.plot(year_summary['year'], year_summary['total_streams_millions'], marker='o')
plt.title('Total Streams by Year')
plt.xlabel('Year')
plt.ylabel('Total Streams in Millions')
plt.show()


Analysis of Streams by Year

The graph shows how total streams change across release years in the dataset. Some years have higher total streams because more major songs from those years are included. This shows that yearly streaming totals depend on both the number of songs in the dataset and the strength of the songs released in that year.


In [ ]:
#Analyzing the Relationship Between Spotify Playlists and Streams
#Calculating the correlation between playlist placement and streams
playlist_stream_corr = df['in_spotify_playlists'].corr(df['streams_millions'])
print('Correlation between Spotify playlists and streams:', round(playlist_stream_corr, 3))

#Display songs with the most Spotify playlist placements
playlist_table = (
    df[['song', 'artist', 'in_spotify_playlists', 'streams_millions', 'popularity']]
    .sort_values('in_spotify_playlists', ascending=False)
    .head(10)
)

display(playlist_table)

#Create a scatter plot comparing playlists and streams
plt.figure(figsize=(10,6))
plt.scatter(df['in_spotify_playlists'], df['streams_millions'], alpha=0.75)
plt.title('Spotify Playlists vs. Streams')
plt.xlabel('Number of Spotify Playlists')
plt.ylabel('Streams in Millions')
plt.show()


Analysis of Spotify Playlist Placement and Streams

The scatter plot compares the number of Spotify playlists a song appears in with its total streams. Playlist placement matters because playlists can increase discovery and repeated listening. If the correlation is positive, that suggests songs in more playlists also tend to have more streams, supporting the idea that platform visibility is connected to streaming success.


In [ ]:
#Analyzing Audio Features and Popularity
#Calculating correlations between selected audio features and popularity
audio_features = [
    'streams_millions',
    'in_spotify_playlists',
    'bpm',
    'energy',
    'danceability',
    'liveness',
    'valence',
    'duration',
    'acousticness',
    'speechiness',
    'popularity'
]

correlation_table = df[audio_features].corr().round(2)
display(correlation_table)

#Scatter plot for danceability and popularity
plt.figure(figsize=(10,6))
plt.scatter(df['danceability'], df['popularity'], alpha=0.75)
plt.title('Danceability vs. Popularity')
plt.xlabel('Danceability')
plt.ylabel('Popularity')
plt.show()

#Scatter plot for energy and popularity
plt.figure(figsize=(10,6))
plt.scatter(df['energy'], df['popularity'], alpha=0.75)
plt.title('Energy vs. Popularity')
plt.xlabel('Energy')
plt.ylabel('Popularity')
plt.show()


Analysis of the Correlation Between Audio Features and Popularity

The correlation table and scatter plots show that audio features help describe songs, but they do not fully explain popularity by themselves. Danceability and energy can be useful for understanding the style of a song, but a song can still become popular even if it is not the most danceable or highest-energy track. This supports the idea that Spotify success comes from a mix of sound, playlist exposure, artist recognition, and replay value.


My Final Insights and Conclusion

Through my analysis of the Spotify songs dataset, several key patterns emerged relating to the six questions I initially proposed:

Top Songs: The most-streamed songs show that Spotify success can last across multiple years. A song does not have to be the newest release to remain highly streamed.

Artists: Artists with multiple popular songs have an advantage because total streams build across their catalog. This shows that artist recognition and consistency matter.

Genres: Some genres have higher average popularity, but genre results should be interpreted with song count. A small genre can rank highly if it contains one major hit.

Years: Stream totals change by year, but those totals depend on how many songs from each year are included and how successful those songs are.

Playlist Placement: Songs that appear in more Spotify playlists may have stronger streaming performance because playlists increase visibility and discovery.

Audio Features: Danceability and energy help describe a song's style, but they do not fully predict popularity. Overall, the data suggests that Spotify success is a combination of platform visibility, artist strength, replay value, and song characteristics.
